# EDDI With xarray

This notebook computes EDDI from a labeled monthly PET series using the v2.5 typed public API.

In [1]:
import logging

import numpy as np
import pandas as pd
import xarray as xr

from climate_indices import eddi

logging.disable(logging.CRITICAL)

In [2]:
time = pd.date_range("2000-01-01", periods=10 * 12, freq="MS")
month_index = np.arange(time.size)
pet = xr.DataArray(
    50.0 + 10.0 * np.sin(2.0 * np.pi * month_index / 12.0) + 2.0 * np.cos(month_index / 3.0),
    coords={"time": time},
    dims="time",
    name="pet",
    attrs={"units": "mm", "long_name": "Monthly potential evapotranspiration"},
)

eddi_3 = eddi(
    pet,
    scale=3,
    calibration_year_initial=2000,
    calibration_year_final=2009,
)

eddi_3

<xarray.DataArray 'pet' (time: 120)> Size: 960B
array([            nan,             nan,  9.91558895e-01,  6.50746449e-01,
        3.73271803e-01,  1.22573284e-01, -1.20132646e-01, -3.70670917e-01,
       -6.47747776e-01, -9.87596032e-01, -1.51551561e+00, -1.51551561e+00,
       -9.19112585e-01, -5.64221810e-01, -3.70670917e-01, -1.20132646e-01,
        1.22573284e-01,  3.73271803e-01,  6.50746449e-01,  1.52321149e+00,
        1.52321149e+00,  9.91558895e-01,  6.50746449e-01,  6.50746449e-01,
        2.72949458e-01, -2.70162907e-01, -6.47747776e-01, -9.87596032e-01,
       -1.51551561e+00, -1.51551561e+00, -1.51551561e+00, -6.47747776e-01,
       -3.70670917e-01, -1.20132646e-01,  1.22573284e-01,  3.73271803e-01,
        5.67377380e-01,  1.47058976e+00,  1.52321149e+00,  1.52321149e+00,
        9.91558895e-01,  6.50746449e-01,  3.73271803e-01, -1.20132646e-01,
       -1.20132646e-01, -3.70670917e-01, -6.47747776e-01, -9.87596032e-01,
       -1.46271314e+00, -9.19112585e-01, -9.87596032e-01, -6.47747776e-01,
       -3.70670917e-01, -1.20132646e-01,  1.22573284e-01,  6.50746449e-01,
        9.91558895e-01,  1.52321149e+00,  1.52321149e+00,  9.91558895e-01,
        9.23222885e-01,  2.72949458e-01, -1.20132646e-01, -3.70670917e-01,
       -6.47747776e-01, -9.87596032e-01, -9.87596032e-01, -1.51551561e+00,
       -9.87596032e-01, -6.47747776e-01, -3.70670917e-01, -1.20132646e-01,
        1.33923450e-03,  5.67377380e-01,  6.50746449e-01,  9.91558895e-01,
        1.52321149e+00,  1.52321149e+00,  9.91558895e-01,  3.73271803e-01,
        1.22573284e-01,  1.22573284e-01, -1.20132646e-01, -3.70670917e-01,
       -5.64221810e-01, -1.46271314e+00, -1.51551561e+00, -1.51551561e+00,
       -9.87596032e-01, -6.47747776e-01, -3.70670917e-01,  1.22573284e-01,
        3.73271803e-01,  6.50746449e-01,  9.91558895e-01,  1.52321149e+00,
        1.47058976e+00,  9.23222885e-01,  3.73271803e-01,  1.22573284e-01,
       -1.20132646e-01, -3.70670917e-01, -6.47747776e-01, -9.87596032e-01,
       -1.51551561e+00, -1.51551561e+00, -9.87596032e-01, -6.47747776e-01,
       -2.70162907e-01,  1.33923450e-03,  1.22573284e-01,  3.73271803e-01,
        6.50746449e-01,  9.91558895e-01,  1.52321149e+00,  9.91558895e-01,
        6.50746449e-01,  3.73271803e-01,  3.73271803e-01,  1.22573284e-01])
Coordinates:
  * time     (time) datetime64[us] 960B 2000-01-01 2000-02-01 ... 2009-12-01
Attributes:
    units:                     dimensionless
    long_name:                 Evaporative Demand Drought Index
    references:                Hobbins, M. T., Wood, A., McEvoy, D. J., Hunti...
    scale:                     3
    calibration_year_initial:  2000
    calibration_year_final:    2009
    climate_indices_version:   2.4.0
    history:                   2026-05-14T20:29:35Z: EDDI-3 calculated (clima...

In [3]:
assert eddi_3.dims == pet.dims
assert eddi_3.time.identical(pet.time)
assert eddi_3.attrs["long_name"] == "Evaporative Demand Drought Index"
assert eddi_3.attrs["units"] == "dimensionless"
assert np.nanmin(eddi_3.values) >= -3.09
assert np.nanmax(eddi_3.values) <= 3.09

eddi_3.to_series().dropna().head()

time
2000-03-01    0.991559
2000-04-01    0.650746
2000-05-01    0.373272
2000-06-01    0.122573
2000-07-01   -0.120133
Freq: MS, Name: pet, dtype: float64